# Thesis supervision letters from Google Sheets / Drive links

1. **Run the cell.** It asks for `Thesis file link 1`, then `link 2`, and so on. Paste one link per prompt and press **Enter on an empty line** when you're done. A real Google Sheet and an `.xlsx`/`.xls`/`.csv` stored in Drive both work. Nothing is uploaded to or written in your Drive.
   - **Private file:** Colab shows a Google sign-in pop-up once and reads the file with your own account.
   - **Shared as "Anyone with the link":** downloads directly, no sign-in.
   - The same link twice, or two links to identical copies, are only used once.
2. Paste the link of the workbook with the **Employee ID** sheet, or press Enter to skip.
3. All files are **appended into one file**, `letters/combined_thesis_distribution.xlsx`: one header row, one faculty column per semester (oldest first), and Source File / Source Row columns showing where each student came from. **The letters are made from this combined file.**
4. Pick the semester(s). Only that semester's faculty column is used. Then enter the date and the reference prefix.
5. `letters.zip` downloads with `letters/<Semester>/<Faculty>.pdf` and the combined file.

To skip the questions, fill `THESIS_SOURCES` and `EMPLOYEE_SOURCE` at the top of the code.

In [ ]:
import csv
import hashlib
import io
import os
import re
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
from difflib import get_close_matches

for _mod, _pkg in (("reportlab", "reportlab"), ("openpyxl", "openpyxl"), ("xlrd", "xlrd")):
    try:
        __import__(_mod)
    except ImportError:
        print(f"{_pkg} not found -- installing it now (needs internet access)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import openpyxl
import xlrd   # only for old .xls files; .xlsx/.xlsm go through openpyxl
from openpyxl.styles import Font
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (
    Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, KeepTogether
)

# No Drive mount: input files are read straight from their Google links.
try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------------------
# 0. Config -- EDIT THESE for your environment
# ---------------------------------------------------------------------------
# Google Sheets / Google Drive link(s) of the thesis distribution file(s).
# Leave the list empty and the notebook asks you to paste them when it runs.
# A link can point to a real Google Sheet or to an .xlsx / .xls / .csv file
# sitting in Drive. Nothing is uploaded to or written in your Drive: the file
# is only downloaded into this notebook session. Local file paths work too.
#   - File shared as "Anyone with the link": downloads directly, anywhere.
#   - Private file: in Google Colab you get one Google sign-in pop-up and it
#     is read with your own account (no sharing needed).
# In Excel files EVERY sheet is checked; sheets that aren't a thesis roster
# are skipped. Columns are matched by HEADER NAME, never by position.
THESIS_SOURCES = [
    # "https://docs.google.com/spreadsheets/d/XXXXXXXX/edit",
]

# Link (or path) of the workbook holding the faculty -> Employee ID list,
# e.g. the old 'spring & summer.xlsx'. "" = ask when the notebook runs
# (press Enter there to skip). Any sheet with an 'Employee ID' column inside
# the thesis files is also picked up automatically.
EMPLOYEE_SOURCE = ""
EMPLOYEE_SHEET_NAME = "Employee ID"   # used when the roster is an Excel file

DOWNLOAD_DIR = "downloaded_inputs"    # session-local copy of the linked files

OUT_DIR = "/kaggle/working/letters" if os.path.isdir("/kaggle/working") else "letters"
FIXED_CR = 1.0
FIXED_HRS = 1.0

# All linked files are appended into this one workbook first, and the
# letters are made from it. It goes into the letters zip so you can check it.
COMBINED_FILE = os.path.join(OUT_DIR, "combined_thesis_distribution.xlsx")

# Accepted header spellings (compared after lower-casing and dropping
# everything that is not a letter or digit, so "Student ID", "student_id"
# and "STUDENT-ID" all become "studentid").
COLUMN_ALIASES = {
    "Group":    ["group", "groupid", "groupno", "groupnumber", "grp", "grpno"],
    "ID":       ["id", "studentid", "stdid", "sid", "studentidno"],
    "Name":     ["name", "studentname", "nameofstudent", "nameofthestudent"],
    "Batch":    ["batch", "batchno", "batchnumber"],
    "Faculty":  ["faculty", "facultyname", "assignedfaculty", "supervisor",
                 "supervisorname", "facultymember"],
    "Semester": ["semester", "semestername", "session", "term"],
    # only in the combined file: where each row originally came from
    "Source":        ["sourcefile"],
    "SourceSection": ["sourcesection"],
    "SourceRow":     ["sourcerow"],
}
REQUIRED_COLUMNS = ["Group", "ID", "Name", "Batch"]   # + Faculty or semester columns
EMP_ALIASES = {
    "EmpID": ["employeeid", "empid", "employeeidno", "id"],
    "FacName": ["facultyname", "name", "faculty", "employeename", "nameoffaculty"],
}
HEADER_SEARCH_ROWS = 15   # roster header may sit below a few title rows
SEM_LABEL_ROWS_ABOVE = 4  # 'Spring2026' may sit a few rows above 'Assigned Faculty'
INPUT_EXTS = (".csv", ".xlsx", ".xlsm", ".xls")

os.makedirs(OUT_DIR, exist_ok=True)


def s(x):
    """Cell value as a clean string: '' for None/blank, and runs of spaces,
    tabs and line breaks collapsed to one space (cells like
    ' \\n \\t\\nDr. Md. Shakowat Zaman Sarker (DMSZ)\\n' are common)."""
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def safe_filename(name):
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def clean_emp_id(x):
    """Normalize an ID-like cell. Excel stores 150271 as 150271.0 and some CSV
    exports write '150271.0' too -- strip that '.0'. IDs with leading zeros
    (e.g. '090104') are kept exactly as written."""
    if x is None:
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    x = str(x).strip()
    if re.fullmatch(r"\d+\.0+", x):
        x = x.split(".")[0]
    return x


def normalize_name(n):
    """Case/whitespace/punctuation-insensitive key for name matching."""
    n = n.lower().strip()
    n = n.replace(".", "")
    n = re.sub(r"\s+", " ", n)
    return n


def strip_acronym(n):
    """Drop a trailing '(XYZ)' style acronym, e.g. 'Md. Rezaul Karim (RK)'
    -> 'Md. Rezaul Karim'."""
    return re.sub(r"\s*\([^)]*\)\s*$", "", n).strip()


def norm_header(h):
    return re.sub(r"[^a-z0-9]", "", s(h).lower())


SEASONS = {"spring": "Spring", "sp": "Spring", "summer": "Summer", "su": "Summer",
           "fall": "Fall", "fa": "Fall", "fl": "Fall", "autumn": "Autumn",
           "winter": "Winter"}
SEMESTER_RE = re.compile(
    r"(?<![a-z])(spring|summer|fall|autumn|winter|sp|su|fa|fl)[\s_\-]*'?(\d{4}|\d{2})(?!\d)",
    re.I)


def normalize_semester(text):
    """'summer-2026', 'SUMMER 26', 'Summer2026', 'SU26' -> 'Summer 2026'.
    Anything that doesn't look like <season> <year> is kept as written."""
    text = s(text)
    m = SEMESTER_RE.search(text)
    if not m:
        return re.sub(r"\s+", " ", text)
    year = m.group(2)
    if len(year) == 2:
        year = "20" + year
    return f"{SEASONS[m.group(1).lower()]} {year}"


def find_semester(text):
    m = SEMESTER_RE.search(s(text))
    return normalize_semester(m.group(0)) if m else ""


SEASON_ORDER = {"Spring": 0, "Summer": 1, "Fall": 2, "Autumn": 2, "Winter": 3}


def semester_sort_key(sem):
    """'Spring 2026' < 'Summer 2026' < 'Fall 2026' < 'Spring 2027'; anything
    that isn't '<Season> <year>' goes last."""
    m = re.fullmatch(r"(\w+) (\d{4})", sem)
    if m and m.group(1) in SEASON_ORDER:
        return (0, int(m.group(2)), SEASON_ORDER[m.group(1)], sem)
    return (1, 0, 0, sem)


def is_semester_header(h):
    """True for a cell that STARTS with a semester, e.g. 'Spring 26',
    'Summer-2026', 'Fall2026 (Stage 3 of 3)', 'SU26' -- not 'Remarks Spring 26'
    or 'Thesis Distribution, Spring-2026'."""
    return bool(SEMESTER_RE.match(s(h)))


# ---------------------------------------------------------------------------
# 1. File readers -- every input becomes a list of "tables" (one per CSV,
#    one per Excel sheet), each a plain list of rows.
# ---------------------------------------------------------------------------
def read_csv_rows(path):
    """All rows of a CSV as lists of strings. Excel on Windows saves CSVs as
    either UTF-8 with BOM or cp1252, so try both."""
    for enc in ("utf-8-sig", "cp1252"):
        try:
            with open(path, newline="", encoding=enc) as fh:
                return [row for row in csv.reader(fh)]
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode '{path}' as UTF-8 or cp1252.")


def read_xlsx_sheets(path):
    """{sheet_name: rows} for .xlsx/.xlsm. Merged cells are un-merged: every
    cell of a merged block gets the block's value, so a Faculty or Group
    merged over 4 student rows shows up on all 4. A one-row merge across
    several columns is a title banner ('Evening Batch (43 Eve ...)' over
    A:G) and is left in its first cell only -- spreading it would put the
    banner text into the Student ID column."""
    wb = openpyxl.load_workbook(path, data_only=True)
    out = {}
    for ws in wb.worksheets:
        rows = [list(r) for r in ws.iter_rows(values_only=True)]
        for rng in ws.merged_cells.ranges:
            if rng.min_row == rng.max_row:
                continue
            val = ws.cell(rng.min_row, rng.min_col).value
            for r in range(rng.min_row, rng.max_row + 1):
                for c in range(rng.min_col, rng.max_col + 1):
                    if r - 1 < len(rows) and c - 1 < len(rows[r - 1]):
                        rows[r - 1][c - 1] = val
        out[ws.title] = rows
    return out


def read_xls_sheets(path):
    """{sheet_name: rows} for old .xls files, merged cells un-merged the
    same way as read_xlsx_sheets."""
    try:
        book = xlrd.open_workbook(path, formatting_info=True)
    except NotImplementedError:          # some .xls variants lack formatting info
        book = xlrd.open_workbook(path)
    out = {}
    for sh in book.sheets():
        rows = [[sh.cell_value(r, c) for c in range(sh.ncols)] for r in range(sh.nrows)]
        for rlo, rhi, clo, chi in getattr(sh, "merged_cells", []):
            if rhi - rlo == 1:
                continue
            val = rows[rlo][clo]
            for r in range(rlo, rhi):
                for c in range(clo, chi):
                    rows[r][c] = val
        out[sh.name] = rows
    return out


def read_tables(path):
    """[(label, sheet_name, rows)] for any supported file."""
    ext = os.path.splitext(path)[1].lower()
    base = os.path.basename(path)
    if ext == ".csv":
        return [(base, "", read_csv_rows(path))]
    if ext in (".xlsx", ".xlsm"):
        sheets = read_xlsx_sheets(path)
    elif ext == ".xls":
        sheets = read_xls_sheets(path)
    else:
        raise ValueError(f"Unsupported file type: {base}")
    return [(f"{base} [{name}]", name, rows) for name, rows in sheets.items()]


def find_header(rows, aliases, required):
    """Return (header_row_index, {logical_column: col_index}) for the first
    row in the top HEADER_SEARCH_ROWS that contains every required column."""
    for i, row in enumerate(rows[:HEADER_SEARCH_ROWS]):
        normed = [norm_header(c) for c in row]
        col_idx = {}
        for logical, names in aliases.items():
            for j, h in enumerate(normed):
                if h in names:
                    col_idx[logical] = j
                    break
        if all(c in col_idx for c in required):
            return i, col_idx
    return None, None


def match_columns(row, aliases, skip=()):
    """{logical_column: col_index} for the header cells of one row."""
    normed = [norm_header(c) for c in row]
    col_idx = {}
    for logical, names in aliases.items():
        for j, h in enumerate(normed):
            if j not in skip and h in names:
                col_idx[logical] = j
                break
    return col_idx


def thesis_header_at(rows, i):
    """If rows[i] is a thesis header row, return (col_idx, sem_cols), else None.

    Needs Group/ID/Name/Batch plus either a Faculty column or semester
    columns. A semester column is one whose header cell is a semester
    ('Spring 26'), or whose nearest non-blank cell up to SEM_LABEL_ROWS_ABOVE
    rows above is one -- e.g. three 'Assigned Faculty' columns with
    'Spring2026 / Summer2026 / Fall2026' on a row above them."""
    row = rows[i]
    base = match_columns(row, {k: v for k, v in COLUMN_ALIASES.items()
                               if k != "Faculty"})
    if not all(c in base for c in REQUIRED_COLUMNS):
        return None
    sem_cols = {}
    for j, h in enumerate(row):
        if j in base.values():
            continue
        label = h if is_semester_header(h) else None
        if label is None:
            for k in range(i - 1, max(-1, i - 1 - SEM_LABEL_ROWS_ABOVE), -1):
                above = rows[k][j] if j < len(rows[k]) else None
                if s(above):
                    label = above if is_semester_header(above) else None
                    break
        if label is not None:
            sem_cols.setdefault(normalize_semester(label), j)
    col = dict(base)
    col.update(match_columns(row, {"Faculty": COLUMN_ALIASES["Faculty"]},
                             skip=set(sem_cols.values())))
    if "Faculty" in col or sem_cols:
        return col, sem_cols
    return None


# ---------------------------------------------------------------------------
# 2. Employee ID roster
# ---------------------------------------------------------------------------
emp_map = {}           # normalized name (full, incl. acronym) -> employee id
emp_map_stripped = {}  # normalized name with acronym stripped -> employee id


def add_roster_rows(rows, hdr, col):
    n = 0
    for row in rows[hdr + 1:]:
        row = list(row) + [None] * (max(col.values()) + 1)
        emp_id = clean_emp_id(row[col["EmpID"]])
        name = s(row[col["FacName"]])
        if not emp_id or not name:
            continue
        emp_map.setdefault(normalize_name(name), emp_id)
        emp_map_stripped.setdefault(normalize_name(strip_acronym(name)), emp_id)
        n += 1
    return n


def load_employee_file(path, sheet_name):
    if not os.path.isfile(path):
        return
    if path.lower().endswith(".csv"):
        rows = read_csv_rows(path)
    else:
        sheets = (read_xls_sheets(path) if path.lower().endswith(".xls")
                  else read_xlsx_sheets(path))
        if sheet_name not in sheets:
            found = [name for name, rows in sheets.items()
                     if try_load_roster_table(f"{os.path.basename(path)} [{name}]", rows)]
            if not found:
                raise ValueError(
                    f"No '{sheet_name}' sheet (or any sheet with an 'Employee ID' "
                    f"column) in '{os.path.basename(path)}'. Sheets: {list(sheets)}")
            return
        rows = sheets[sheet_name]
    hdr, col = find_header(rows, EMP_ALIASES, ["EmpID", "FacName"])
    if hdr is None:
        hdr, col = 0, {"EmpID": 0, "FacName": 1}   # old layout: A = ID, B = name
    n = add_roster_rows(rows, hdr, col)
    print(f"Loaded {n} employee ID entries from '{os.path.basename(path)}'.")


def try_load_roster_table(label, rows):
    """Use a sheet/CSV as the employee roster if it has an explicit Employee
    ID header (not just 'ID', which every student sheet has too)."""
    strict = {"EmpID": ["employeeid", "empid", "employeeidno"],
              "FacName": EMP_ALIASES["FacName"]}
    hdr, col = find_header(rows, strict, ["EmpID", "FacName"])
    if hdr is None:
        return False
    n = add_roster_rows(rows, hdr, col)
    print(f"  {label}: employee roster, {n} entries")
    return True


def lookup_employee_id(faculty_name):
    key_full = normalize_name(faculty_name)
    if key_full in emp_map:
        return emp_map[key_full]
    key_stripped = normalize_name(strip_acronym(faculty_name))
    if key_stripped in emp_map_stripped:
        return emp_map_stripped[key_stripped]
    if key_stripped in emp_map:
        return emp_map[key_stripped]
    return ""


# ---------------------------------------------------------------------------
# 3. Turn one table into flat student records
# ---------------------------------------------------------------------------
def parse_thesis_table(label, rows, fallback_semester, from_csv):
    """Return (records, mode_description) or ([], None) if this table is not
    a thesis roster.

    A sheet may hold several sections (e.g. 'Day Batch' and 'Evening Batch'),
    each starting with its own header row; every header row found anywhere
    in the sheet starts a new section and resets the forward-fill.

    Semester is decided in this order:
      a) semester columns ('Spring 26', or 'Assigned Faculty' under a
         'Spring2026' label): one record per student per filled-in semester
         column. If the section has no separate Faculty column, the cell
         value IS the faculty for that semester; if it has one, a non-blank
         cell just means "in this semester". Only the column of the semester
         the user picks is ever used.
      b) a 'Semester' column,
      c) the sheet name, then the file name (e.g. 'summer_2026.csv').
    Group / Faculty / Semester are forward-filled down blank cells (what
    merged cells become in a CSV). Excel merged cells are already un-merged
    by the readers."""
    records, modes = [], []
    sections = 0
    col = None

    for i, raw in enumerate(rows):
        line_no = i + 1
        hdr = thesis_header_at(rows, i)
        if hdr:
            col, sem_cols = hdr
            sections += 1
            if not sem_cols and "Semester" not in col and not fallback_semester:
                raise ValueError(
                    f"{label} row {line_no}: no semester columns, no 'Semester' "
                    f"column and no semester in the sheet/file name -- can't "
                    f"tell which semester these students belong to.")
            has_faculty = "Faculty" in col
            if sem_cols:
                mode = (f"semester columns {list(sem_cols)} "
                        + ("(filter; faculty from 'Faculty' column)" if has_faculty
                           else "(cell value = faculty)"))
            elif "Semester" in col:
                mode = "'Semester' column"
            else:
                mode = f"semester from name: {fallback_semester}"
            if mode not in modes:
                modes.append(mode)
            width = max(list(col.values()) + list(sem_cols.values())) + 1
            cur_group, cur_faculty, cur_sem = "", "", fallback_semester
            cur_semval = {sem: "" for sem in sem_cols}
            cur_origin = None
            continue
        if col is None:          # title rows above the first header
            continue

        r = list(raw) + [None] * width
        grp_str = clean_emp_id(r[col["Group"]])
        id_str = clean_emp_id(r[col["ID"]])
        fac_str = s(r[col["Faculty"]]) if has_faculty else ""
        # In the combined file every row says where it came from; elsewhere
        # the origin is this table / section / row.
        origin = (s(r[col["Source"]]) if "Source" in col else label,
                  clean_emp_id(r[col["SourceSection"]]) if "SourceSection" in col
                  else sections)
        src_row = clean_emp_id(r[col["SourceRow"]]) if "SourceRow" in col else line_no
        new_group = ((bool(grp_str) and grp_str != cur_group)
                     or (bool(fac_str) and fac_str != cur_faculty)
                     or origin != cur_origin)
        cur_origin = origin

        if "Semester" in col and not sem_cols:
            sem_str = normalize_semester(r[col["Semester"]])
            if sem_str:
                cur_sem = sem_str
        if fac_str and fac_str != cur_faculty:
            cur_faculty = fac_str
            if not grp_str and id_str:
                print(f"  WARNING {label} row {line_no}: faculty changes to "
                      f"'{fac_str}' but Group is blank -- student put in the "
                      f"previous group '{cur_group}'.")
        if grp_str:
            cur_group = grp_str
        # Blank semester-column cells:
        #  - inside the same group: always inherit (a group can't be half in
        #    a semester);
        #  - at a new group: blank means "not in this semester" -- except in
        #    a CSV where the cell value is the faculty, because there a name
        #    merged over all of a faculty's groups (like the old Faculty
        #    column) leaves blanks after export. Excel merges are already
        #    un-merged by the readers, so Excel never needs that.
        for sem, j in sem_cols.items():
            v = s(r[j])
            if v:
                cur_semval[sem] = v
            elif new_group and (has_faculty or not from_csv):
                cur_semval[sem] = ""

        # Rows with no Student ID are blank separator rows or section titles
        # (e.g. "Eve Batch 44 (...)") -- skip both.
        if not id_str:
            continue
        if re.fullmatch(r"\d+(\.\d+)?[eE]\+\d+", id_str):
            print(f"  WARNING {label} row {line_no}: Student ID '{id_str}' is in "
                  f"scientific notation -- Excel mangled it. Format the ID "
                  f"column as Text before saving the CSV.")

        base = {
            "Group": cur_group,
            "ID": id_str,
            "Name": s(r[col["Name"]]),
            "Batch": clean_emp_id(r[col["Batch"]]),
            "Source": origin[0],
            "Section": origin[1],
            "Line": src_row,
        }
        if sem_cols:
            for sem in sem_cols:
                v = cur_semval[sem]
                if not v:
                    continue
                faculty = cur_faculty if has_faculty else v
                if faculty:
                    records.append(dict(base, Semester=sem, Faculty=faculty))
        elif cur_faculty:
            records.append(dict(base, Semester=cur_sem or "(no semester)",
                                Faculty=cur_faculty))

    if not sections:
        return [], None
    return records, f"{sections} section(s); " + "; ".join(modes)


# ---------------------------------------------------------------------------
# 4. Get the input files. Google links are downloaded into DOWNLOAD_DIR
#    (local to this session -- nothing goes to your Drive); local paths are
#    used as they are.
# ---------------------------------------------------------------------------
XLSX_MIME = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_drive_service = None


def is_input_file(p):
    b = os.path.basename(p)
    return b.lower().endswith(INPUT_EXTS) and not b.startswith("~$")


def drive_file_id(text):
    """File ID from a Google Sheets / Drive link (.../d/<ID>/edit...,
    ...open?id=<ID>, ...uc?id=<ID>), or a bare file ID. '' if none."""
    text = text.strip()
    m = (re.search(r"/d/([A-Za-z0-9_-]{20,})", text)
         or re.search(r"[?&]id=([A-Za-z0-9_-]{20,})", text))
    if m:
        return m.group(1)
    return text if re.fullmatch(r"[A-Za-z0-9_-]{25,}", text) else ""


def detect_ext(data):
    """File type from the downloaded bytes; '' for an HTML page (what Google
    sends back instead of the file when it isn't shared publicly)."""
    if data[:4] == b"PK\x03\x04":
        return ".xlsx"
    if data[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":
        return ".xls"
    head = data[:1024].lstrip().lower()
    if not head or head.startswith((b"<!doctype html", b"<html")):
        return ""
    return ".csv"


def download_public(fid):
    """Works only for files shared as 'Anyone with the link'. A real Google
    Sheet comes through the export URL, an uploaded .xlsx/.xls/.csv through
    Drive's download URL. Returns (bytes, filename) or (None, '')."""
    for url in (f"https://docs.google.com/spreadsheets/d/{fid}/export?format=xlsx",
                f"https://drive.google.com/uc?export=download&id={fid}"):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
                disp = resp.headers.get("Content-Disposition") or ""
        except Exception:
            continue
        if detect_ext(data):
            m = (re.search(r"filename\*=UTF-8''([^;]+)", disp)
                 or re.search(r'filename="([^"]+)"', disp))
            return data, urllib.parse.unquote(m.group(1)) if m else ""
    return None, ""


def download_with_drive_api(fid):
    """Colab only: sign in as you (one pop-up, first time only) and read the
    file through the Drive API, so it does NOT have to be shared."""
    global _drive_service
    if _drive_service is None:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        _drive_service = build("drive", "v3")
    from googleapiclient.http import MediaIoBaseDownload

    meta = _drive_service.files().get(
        fileId=fid, fields="name,mimeType", supportsAllDrives=True).execute()
    name = meta.get("name") or fid
    if meta.get("mimeType") == "application/vnd.google-apps.spreadsheet":
        request = _drive_service.files().export_media(fileId=fid, mimeType=XLSX_MIME)
        name += ".xlsx"
    else:
        request = _drive_service.files().get_media(fileId=fid, supportsAllDrives=True)
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return buf.getvalue(), name


def fetch_source(src):
    """Local path -> itself. Google Sheets/Drive link -> path of a
    downloaded copy in DOWNLOAD_DIR."""
    src = src.strip()
    if os.path.isfile(src):
        return src
    fid = drive_file_id(src)
    if not fid:
        raise ValueError(f"'{src}' is neither an existing file nor a Google "
                         f"Sheets / Drive link.")
    data, name = download_public(fid)
    if data is None and IN_COLAB:
        print("  not shared publicly -- signing in to Google to read it with "
              "your account...")
        try:
            data, name = download_with_drive_api(fid)
        except Exception as e:
            raise RuntimeError(
                f"Your Google account can't open {src}\n  ({e})\n  Sign in with "
                f"the account that owns the file, or ask the owner to share it "
                f"with you.") from None
    if data is None:
        raise RuntimeError(
            f"Could not download {src}\n  Share it as 'Anyone with the link' "
            f"(Share -> General access), or run this notebook in Google Colab "
            f"so it can sign in to your Google account.")
    ext = detect_ext(data)
    if not ext:
        raise RuntimeError(f"{src} did not return a spreadsheet file.")
    base = re.sub(r'[\\/:*?"<>|]+', "_", os.path.splitext(name)[0]).strip() or fid
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    path = os.path.join(DOWNLOAD_DIR, base + ext)
    if os.path.exists(path):
        path = os.path.join(DOWNLOAD_DIR, f"{base}_{fid[:6]}{ext}")
    with open(path, "wb") as fh:
        fh.write(data)
    print(f"  downloaded -> {os.path.basename(path)}")
    return path


def ask_sources():
    """Ask for the links one at a time (Link 1, Link 2, ...); an empty
    answer finishes. Several links pasted on one line also work."""
    sources = []
    while True:
        prompt = (f"Thesis file link {len(sources) + 1} (press Enter when done): "
                  if sources else "Thesis file link 1: ")
        raw = input(prompt).strip()
        if not raw:
            if sources:
                return sources
            continue
        links = re.findall(r"https?://[^\s,]+", raw)
        sources += links or [p.strip() for p in raw.split(",") if p.strip()]


def write_combined_file(records, path):
    """Append every student row of every file into ONE clean sheet: one
    header row, no merged cells, one faculty column per semester (oldest
    semester first), and Source File / Source Section / Source Row so each
    row can be traced back to the sheet it came from.
    Returns (number_of_rows, semesters)."""
    semesters = sorted({r["Semester"] for r in records}, key=semester_sort_key)
    rows = {}
    for rec in records:
        key = (rec["Source"], rec["Section"], rec["Line"])
        row = rows.setdefault(key, {"Group": rec["Group"], "ID": rec["ID"],
                                    "Name": rec["Name"], "Batch": rec["Batch"],
                                    "Faculty": {}})
        row["Faculty"][rec["Semester"]] = rec["Faculty"]

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Combined"
    header = ["Source File", "Source Section", "Source Row", "Group ID",
              "Student ID", "Name", "Batch"] + semesters
    ws.append(header)
    for (src, sec, line), row in rows.items():
        ws.append([src, sec, line, row["Group"], row["ID"], row["Name"],
                   row["Batch"]] + [row["Faculty"].get(sem, "") for sem in semesters])
    for cell in ws[1]:
        cell.font = Font(bold=True)
    for col_letter in ("D", "E", "G"):      # IDs stay text: keeps leading zeros
        for cell in ws[col_letter][1:]:
            cell.number_format = "@"
    widths = [34, 9, 9, 11, 13, 34, 8] + [36] * len(semesters)
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[openpyxl.utils.get_column_letter(i)].width = w
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    wb.save(path)
    return len(rows), semesters


thesis_sources = list(THESIS_SOURCES) or ask_sources()
employee_source = EMPLOYEE_SOURCE.strip() or input(
    "Paste the link of the Employee ID workbook (Enter to skip): ").strip()

print("\nGetting the files:")
input_paths = []
seen_sources, seen_content = {}, {}
for n, src in enumerate(thesis_sources, start=1):
    print(f"  Link {n}: {src}")
    src_key = (os.path.abspath(src.strip()) if os.path.isfile(src.strip())
               else drive_file_id(src) or src.strip())
    if src_key in seen_sources:
        print(f"    SKIPPED: same file as link {seen_sources[src_key]} -- not added twice.")
        continue
    seen_sources[src_key] = n
    path = fetch_source(src)
    if not is_input_file(path):
        raise ValueError(f"{path}: only .csv, .xlsx, .xlsm and .xls are supported.")
    with open(path, "rb") as fh:
        digest = hashlib.sha256(fh.read()).hexdigest()
    if digest in seen_content:
        print(f"    SKIPPED: exactly the same content as link {seen_content[digest]} "
              f"(a copy of the same file) -- not added twice.")
        continue
    seen_content[digest] = n
    input_paths.append(path)

emp_abs = ""
if employee_source:
    print(f"  Employee ID workbook: {employee_source}")
    emp_path = fetch_source(employee_source)
    emp_abs = os.path.abspath(emp_path)
    load_employee_file(emp_path, EMPLOYEE_SHEET_NAME)

source_records = []
print(f"\nReading {len(input_paths)} file(s):")
for p in input_paths:
    if os.path.abspath(p) == emp_abs:
        continue
    for label, sheet_name, rows in read_tables(p):
        fallback = find_semester(sheet_name) or find_semester(os.path.basename(p))
        recs, mode = parse_thesis_table(label, rows, fallback,
                                        p.lower().endswith(".csv"))
        if mode is None:
            if not try_load_roster_table(label, rows):
                print(f"  SKIPPED {label}: no Group/ID/Name/Batch + Faculty "
                      f"header found")
            continue
        n_students = len({(r["Section"], r["Line"]) for r in recs})
        sems = sorted({r["Semester"] for r in recs}, key=semester_sort_key)
        print(f"  {label}: {n_students} students  [{mode}]  "
              f"semester(s): {', '.join(sems) or '-'}")
        source_records.extend(recs)

if not source_records:
    raise ValueError("No student rows found in any file -- see the SKIPPED "
                     "reasons above.")

# Append everything into one file, then make the letters FROM that file.
n_rows, combined_sems = write_combined_file(source_records, COMBINED_FILE)
print(f"\nCombined file: {COMBINED_FILE}  ({n_rows} students from "
      f"{len(input_paths)} file(s); semester columns: {', '.join(combined_sems)})")

all_records = []
for label, sheet_name, rows in read_tables(COMBINED_FILE):
    recs, _ = parse_thesis_table(label, rows, "", False)
    all_records.extend(recs)


def _fingerprint(recs):
    return sorted((r["Source"], str(r["Section"]), str(r["Line"]), r["Semester"],
                   r["Faculty"], r["ID"], r["Group"]) for r in recs)


if _fingerprint(all_records) != _fingerprint(source_records):
    raise RuntimeError(
        f"The combined file does not match the input files "
        f"({len(all_records)} vs {len(source_records)} student-semester rows) "
        f"-- not making letters from it. Please report this.")

if not emp_map:
    print("WARNING: no employee roster found -- every letter will get the "
          "'____' Employee ID placeholder.")

# ---------------------------------------------------------------------------
# 5. Let the user pick the semester(s)
# ---------------------------------------------------------------------------
sem_counts = {}
for r in all_records:
    sem_counts[r["Semester"]] = sem_counts.get(r["Semester"], 0) + 1
semesters = sorted(sem_counts, key=semester_sort_key)

print("\nSemesters found:")
for i, sem in enumerate(semesters, start=1):
    print(f"  {i}. {sem:25s} ({sem_counts[sem]} students)")

while True:
    choice = input("Select semester(s) -- number(s) like '1' or '1,3', or 'all': ").strip()
    if choice.lower() == "all":
        selected = semesters
        break
    try:
        picks = [int(c) for c in re.split(r"[,\s]+", choice) if c]
        if picks and all(1 <= n <= len(semesters) for n in picks):
            selected = [semesters[n - 1] for n in dict.fromkeys(picks)]
            break
    except ValueError:
        pass
    print(f"  Invalid choice '{choice}'. Enter numbers between 1 and {len(semesters)}.")

LETTER_DATE = input("Enter Letter Date (e.g., 10/05/2026): ")
REF_PREFIXES = {}
for sem in selected:
    REF_PREFIXES[sem] = input(
        f"Enter Reference Prefix for {sem} (e.g., UU/EEE/Thesis/OL/SU26/): ")


# ---------------------------------------------------------------------------
# 6. Collapse rows into groups, per faculty, for one semester
# ---------------------------------------------------------------------------
def build_faculty_groups(records):
    """Return (faculty_order, faculty_map). Groups are keyed by (source,
    section, group no) so 'Group 1' in two different files/sections never
    gets merged into one group by accident. The same faculty spelled
    'Tanzim Afdol(TAF)' in one cell and 'Tanzim Afdol (TAF)' in another is
    one faculty (first spelling is used on the letter).
    Same student ID + same name twice = duplicate row, kept once.
    Same student ID + DIFFERENT names = typo in the sheet: both kept, warned."""
    faculty_order = []
    faculty_map = {}
    faculty_display = {}
    seen_ids = {}
    n_dup = 0

    for rec in records:
        first = seen_ids.get(rec["ID"])
        if first is None:
            seen_ids[rec["ID"]] = rec
        elif normalize_name(first["Name"]) == normalize_name(rec["Name"]):
            n_dup += 1
            if n_dup <= 10:
                print(f"  WARNING duplicate student {rec['ID']} ({rec['Name']}): "
                      f"{rec['Source']} row {rec['Line']} ignored, already in "
                      f"{first['Source']} row {first['Line']} under {first['Faculty']}.")
            continue
        else:
            print(f"  WARNING Student ID {rec['ID']} is used for two DIFFERENT "
                  f"students: '{first['Name']}' ({first['Source']} row "
                  f"{first['Line']}, {first['Group']}) and '{rec['Name']}' "
                  f"({rec['Source']} row {rec['Line']}, {rec['Group']}). Both "
                  f"kept -- one ID is probably a typo, fix it in the sheet.")

        canon = re.sub(r"\s*\(", " (", normalize_name(rec["Faculty"]))
        f = faculty_display.setdefault(canon, rec["Faculty"])
        if f not in faculty_map:
            faculty_map[f] = []
            faculty_order.append(f)

        groups = faculty_map[f]
        key = (rec["Source"], rec["Section"], rec["Group"])
        match = next((g for g in groups if g["Key"] == key), None)
        if match is None:
            groups.append({
                "Key": key,
                "GroupNo": rec["Group"],
                "IDs": [rec["ID"]],
                "Names": [rec["Name"]],
                "Batch": [rec["Batch"]],
            })
        else:
            match["IDs"].append(rec["ID"])
            match["Names"].append(rec["Name"])
            match["Batch"].append(rec["Batch"])

    if n_dup > 10:
        print(f"  ... and {n_dup - 10} more duplicate rows ignored "
              f"({n_dup} in total).")
    return faculty_order, faculty_map


# ---------------------------------------------------------------------------
# 7. PDF generation (one file per faculty) -- layout unchanged
# ---------------------------------------------------------------------------
def build_styles(table_font_size, body_font_size):
    return {
        "title": ParagraphStyle("title", fontName="Times-Bold", fontSize=12,
                                 alignment=TA_CENTER, leading=14),
        "title_u": ParagraphStyle("title_u", fontName="Times-Bold", fontSize=12,
                                   alignment=TA_CENTER, leading=14),
        "body": ParagraphStyle("body", fontName="Times-Roman", fontSize=body_font_size,
                                leading=body_font_size + 3),
        "cell": ParagraphStyle("cell", fontName="Times-Roman", fontSize=table_font_size,
                            alignment=TA_CENTER, leading=table_font_size),
        "cell_left": ParagraphStyle("cell_left", fontName="Times-Roman", fontSize=table_font_size,
                                alignment=TA_LEFT, leading=table_font_size),
        "cell_header": ParagraphStyle("cell_header", fontName="Times-Bold", fontSize=table_font_size,
                                  alignment=TA_CENTER, leading=table_font_size),
        "sig": ParagraphStyle("sig", fontName="Times-Bold", fontSize=body_font_size,
                               alignment=TA_CENTER, leading=body_font_size + 3),
    }


def generate_faculty_pdf(faculty_name, groups, emp_id, out_path,
                         semester_text, ref_prefix, letter_date):
    n_students = sum(len(g["IDs"]) for g in groups)

    if n_students <= 20:
        table_font_size, body_font_size = 9, 10
    elif n_students <= 30:
        table_font_size, body_font_size = 8, 9
    else:
        table_font_size, body_font_size = 7, 9

    styles = build_styles(table_font_size, body_font_size)

    doc = SimpleDocTemplate(
        out_path, pagesize=A4,
        leftMargin=0.6 * 72, rightMargin=0.6 * 72,
        topMargin=0.5 * 72, bottomMargin=0.5 * 72,
    )
    usable_width = doc.width

    story = []
    story.append(Paragraph("Department of Electrical and Electronic Engineering", styles["title"]))
    story.append(Paragraph("Uttara University", styles["title"]))
    story.append(Paragraph("<u>Claim of Thesis/Project Supervision</u>", styles["title_u"]))
    story.append(Spacer(1, 8))

    ref_table = Table(
        [[Paragraph(f"Ref: {ref_prefix}", styles["body"]),
          Paragraph(f"Date: {letter_date}", styles["body"])]],
        colWidths=[usable_width * 0.65, usable_width * 0.3],
    )
    ref_table.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.75, colors.black),
        ("INNERGRID", (0, 0), (-1, -1), 0.75, colors.black),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
    ]))
    story.append(ref_table)
    story.append(Spacer(1, 8))

    story.append(Paragraph("To,", styles["body"]))
    emp_id_text = emp_id if emp_id else "____"
    story.append(Paragraph(
        f"{faculty_name}" + "&nbsp;" * 12 + f"Employee ID: {emp_id_text}", styles["body"]))
    story.append(Paragraph(
        f"Sub: Invitation letter to conduct Thesis/Project (EEE 4000) for {semester_text}",
        styles["body"]))
    story.append(Spacer(1, 6))
    story.append(Paragraph("Dear Sir/Madam,", styles["body"]))
    story.append(Spacer(1, 6))

    intro = (
        "We are pleased to inform you that, you are nominated to conduct the course of "
        "&quot;Thesis/Project (EEE 4000)&quot; for " + semester_text + ". The following "
        "students will be guided under your supervision in the concerned department as "
        "per above schedule."
    )
    story.append(Paragraph(intro, styles["body"]))
    story.append(Spacer(1, 8))

    # ---- Table: one physical row per STUDENT, with Sl.No/Group No/Cr./Hrs.
    #      genuinely merged (SPAN) across each group's row range.
    headers = ["Sl.No", "Group No", "Cr.", "Hrs.", "Student ID", "Student Name", "Batch"]
    table_data = [[Paragraph(h, styles["cell_header"]) for h in headers]]

    span_cmds = []
    total_cr = 0.0
    total_hrs = 0.0
    row_idx = 1  # row 0 is the header

    for g_idx, g in enumerate(groups, start=1):
        n_rows = len(g["IDs"])
        total_cr += FIXED_CR
        total_hrs += FIXED_HRS
        start_row = row_idx

        for r in range(n_rows):
            sl_txt = str(g_idx) if r == 0 else ""
            grp_txt = g["GroupNo"] if r == 0 else ""
            cr_txt = f"{FIXED_CR:.1f}" if r == 0 else ""
            hrs_txt = f"{FIXED_HRS:.1f}" if r == 0 else ""
            table_data.append([
                Paragraph(sl_txt, styles["cell"]),
                Paragraph(grp_txt, styles["cell"]),
                Paragraph(cr_txt, styles["cell"]),
                Paragraph(hrs_txt, styles["cell"]),
                Paragraph(g["IDs"][r], styles["cell"]),
                Paragraph(g["Names"][r].upper(), styles["cell_left"]),
                Paragraph(g["Batch"][r], styles["cell"]),
            ])
            row_idx += 1

        end_row = row_idx - 1
        if n_rows > 1:
            for col in range(4):  # Sl.No, Group No, Cr., Hrs.
                span_cmds.append(("SPAN", (col, start_row), (col, end_row)))
            span_cmds.append(("VALIGN", (0, start_row), (3, end_row), "MIDDLE"))

    # Total row
    table_data.append([
        Paragraph("<b>Total=</b>", styles["cell"]),
        Paragraph("", styles["cell_header"]),
        Paragraph(f"{total_cr:g}", styles["cell_header"]),
        Paragraph(f"{total_hrs:g}", styles["cell_header"]),
        Paragraph("", styles["cell"]),
        Paragraph("", styles["cell"]),
        Paragraph("", styles["cell"]),
    ])
    total_row_idx = row_idx
    span_cmds.append(("SPAN", (0, total_row_idx), (1, total_row_idx)))
    span_cmds.append(("ALIGN", (1, total_row_idx), (1, total_row_idx), "RIGHT"))
    span_cmds.append(("SPAN", (4, total_row_idx), (6, total_row_idx)))

    span_cmds.append(("LINEBELOW", (4, total_row_idx), (6, total_row_idx), 2, colors.white))
    span_cmds.append(("LINEBELOW", (0, total_row_idx), (1, total_row_idx), 2, colors.white))
    span_cmds.append(("LINEBEFORE", (0, total_row_idx), (1, total_row_idx), 1.1, colors.white))
    span_cmds.append(("LINEAFTER", (6, total_row_idx), (6, total_row_idx), 1.1, colors.white))
    span_cmds.append(("LINEBELOW", (2, total_row_idx), (3, total_row_idx), 1, colors.black))

    col_fractions = (0.08, 0.12, 0.08, 0.08, 0.14, 0.40, 0.10)
    col_widths = [usable_width*.90 * f for f in col_fractions]
    main_table = Table(table_data, colWidths=col_widths, repeatRows=1)
    style_cmds = [
        ("BOX", (0, 0), (-1, -1), 0.75, colors.black),
        ("INNERGRID", (0, 0), (-1, -1), 0.75, colors.black),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("LEFTPADDING", (0, 0), (-1, -1), 2),
        ("RIGHTPADDING", (0, 0), (-1, -1), 2),
        ("TOPPADDING", (0, 0), (-1, -1), 1),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 1),
    ] + span_cmds
    main_table.setStyle(TableStyle(style_cmds))
    story.append(main_table)
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>Total Amount of Claim:&nbsp;&nbsp;Tk:</b>", styles["body"]))
    story.append(Spacer(1, 8))
    story.append(Paragraph("Thanking you very much for your cooperation.", styles["body"]))
    story.append(Paragraph("Yours Sincerely", styles["body"]))
    story.append(Spacer(1, 28))
    story.append(Paragraph("Thesis/Project Convener, Dept. of EEE, Uttara University", styles["body"]))
    story.append(Spacer(1, 28))

    sig_table = Table([[
        Paragraph("<br/>Signature of the Faculty Member<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("<br/>Signature of the Coordinator<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("<br/>Signature of the Chairman<br/>(with date &amp; seal)", styles["sig"]),
    ]], colWidths=[usable_width / 3] * 3)
    sig_table.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP")]))
    story.append(sig_table)
    story.append(Spacer(1, 28))

    sig_table2 = Table([[
        Paragraph("Signature of the Deputy Registrar<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("Signature of the Deputy Registrar<br/>(with date &amp; seal)", styles["sig"]),
    ]], colWidths=[usable_width / 2] * 2)

    checked_by_section = [
        Paragraph("<b>Checked By:</b>", styles["body"]),
        Spacer(1, 28),
        sig_table2,
        Spacer(1, 28)
    ]

    story.append(KeepTogether(checked_by_section))

    doc.build(story)


# ---------------------------------------------------------------------------
# 8. Generate the letters: one folder per selected semester, one PDF per
#    faculty inside it. Only the selected semester's records are used.
# ---------------------------------------------------------------------------
all_unmatched = {}
for sem in selected:
    sem_records = [r for r in all_records if r["Semester"] == sem]
    print(f"\n=== {sem}: {len(sem_records)} student rows ===")
    faculty_order, faculty_map = build_faculty_groups(sem_records)
    print(f"Found {len(faculty_order)} faculty members.")

    sem_dir = os.path.join(OUT_DIR, safe_filename(sem))
    os.makedirs(sem_dir, exist_ok=True)

    for f in faculty_order:
        groups = faculty_map[f]
        n_students = sum(len(g["IDs"]) for g in groups)
        emp_id = lookup_employee_id(f)
        if not emp_id:
            all_unmatched.setdefault(f, []).append(sem)

        # Sanity check: every group having exactly one student usually means
        # the Group column is wrong / not forward-filled.
        if n_students > 1 and len(groups) == n_students:
            print(f"  WARNING {f}: {len(groups)} groups for {n_students} students "
                  f"-- check the Group column for this faculty.")

        out_name = safe_filename(f) + ".pdf"
        generate_faculty_pdf(f, groups, emp_id, os.path.join(sem_dir, out_name),
                             sem, REF_PREFIXES[sem], LETTER_DATE)
        print(f"  {f:45s} {len(groups):2d} group(s), {n_students:3d} student(s)  "
              f"Emp ID: {emp_id or 'NOT FOUND'}  -> {out_name}")

if all_unmatched:
    all_names = list(emp_map)
    print(f"\nWARNING: no Employee ID for {len(all_unmatched)} faculty member(s) -- "
          f"their letters keep the '____' placeholder:")
    for f, sems in all_unmatched.items():
        guess = get_close_matches(normalize_name(f), all_names, n=1, cutoff=0.6)
        hint = f"  (closest roster name: '{guess[0]}')" if guess else ""
        print(f"    - {f}  [{', '.join(sems)}]{hint}")

print(f"\nAll letters written to '{OUT_DIR}' (one sub-folder per semester).")

if IN_COLAB:
    zip_path = shutil.make_archive(
        "letters", "zip", root_dir=os.path.dirname(os.path.abspath(OUT_DIR)),
        base_dir=os.path.basename(os.path.abspath(OUT_DIR)))
    files.download(zip_path)
